<div align="right" style="text-align: right"><i>Peter Norvig<br>Dec 2018<br>Updated Jun 2020</i></div>

# Portmantout Words

A [***portmanteau***](https://en.wikipedia.org/wiki/Portmanteau) is a word that squishes together two words, like  ***mathlete*** = ***math*** + ***athlete***.  Inspired by [**Darius Bacon**](http://wry.me/), I covered this as a programming exercise in my 2012 [**Udacity course**](https://www.udacity.com/course/design-of-computer-programs--cs212). In 2018 I was re-inspired by [**Tom Murphy VII**](http://tom7.org/), who added a new twist:  [***portmantout words***](http://www.cs.cmu.edu/~tom7/portmantout/) ([***tout***](https://www.duolingo.com/dictionary/French/tout/fd4dc453d9be9f32b7efe838ebc87599) from the French for *all*).


Informally, **portmantout** means that you take all the words in a set and mush them together in some order such that each word overlaps the next. More formally, a partmantout of a set of words *W* is a string *S* constructed as follows:
- Make an ordered list *L* that contains all the words in *W*.
- It is allowable for a word to appear more than once in *L*.
- Every word in *L* (except the first) must *overlap*: it must begin with a prefix that is the same as a suffix of the previous word.
- The string *S* is formed by concatenating the letters in *L*, but using each overlap only once, not twice.
- For example, for the set *W* = {world, hello, lowbrow}, we can use *L* = [hel<u>lo</u>, <u>lo</u>wbro<u>w</u>, <u>w</u>orld], giving us *S* = "hellowbroworld".


This notebook develops a program that can find a portmantout *S* for a set *W* of over 100,000 words in a few seconds. The program attempts to minimize the length of *S*, but does not guarantee that it is the shortest possible. Along the way it found these interesting portmanteaux:


- **preferendumdums** [prefer, referendum, dumdums]: agreeable  uninformed voters.
- **fortyphonshore** [forty, typhons, onshore]: a dire weather report. 
- **allegestionstage** [alleges, egestions, onstage]: a brutal theatre critique.
- **skymanipulablearsplittingler** [skyman, manipulable, blears, earsplitting, tinglers]: a nerve-damaging aviator.
- **edinburgherselflesslylyricize** [edinburgh, burghers, herself, selflessly, slyly, lyricize]: a Scottish music review.
- **impromptutankhamenability** [impromptu, tutankhamen, amenability]: willingness to see the Egyptian exhibit on the spur of the moment.
- **dashikimonogrammarianarchy** [dashiki, kimono, monogram, grammarian, anarchy]: the chaos that ensues when a linguist gets involved in choosing how to enscribe African/Japanese garb. 

# Problem-Solving Strategy

I originally thought I would define a major function, `portman`, to generate the portmantout string *S* from the set of words *W*, and a minor function, `is_portman`, to verify the result:

    portman(W: Wordset) -> str              # Compute the string S from the set of words W
    is_portman(S: str, W: Wordset) -> bool  # Verify that S is a valid portmantout covering W

But then I realized that verification would be difficult. For example,  *S* =  `'helloworld'` would be rejected as non-overlapping if parsed as `'hello'` + `'world'`, but accepted if parsed as `'hello'` + `'low'` + `'world'`. It was hard for `is_portman` to decide which parse was intended, which is a shame because `portman` *knew* which was intended, as it built up the list of words, *L*. 

To make everything explicit (and more efficient), I will construct not just a **list** of words *L*, but rather what I'll call a **path**, where each **step** in the path *P* consists of a word from *W* and an integer that specifies the number of characters in the overlap with the previous word. With that in mind, I decided on the following calling and [naming](https://en.wikipedia.org/wiki/Natalie_Portman) conventions:

    natalie(W: Wordset) -> Path             # Find a portmantout path P for a set of words W
    portman(P: Path) -> str                 # Compute the string S represented by the path P
    is_portman(P: Path, W: Wordset) -> bool # Verify that P is a valid path covering W

Thus I can generate a portmantout string *S* with:

    S = portman(natalie(W))
    
I distinguish two types of steps:

- **Unused word step**: using a word from *W* for the first time. The unused word must have a prefix that matches the suffix of the previous word. 
- **Bridging word step**: if no unused word overlaps the previous word, we need to do something to get back on track. I call that something a **bridge**: a step that repeats a previously-used word in order to provide a new suffix that will match the prefix of some unused word.  Sometimes there is no such single word, and multiple words are required to build a bridge to an unused word. (On our large word set we never need more than a two-word bridge.)

My intuition is that finding a shortest *S* is an NP-hard problem, and with 100,000 words to cover, it is unlikely that I can find a guaranteed shortest possible string in a reasonable amount of time. A common approach to NP-hard problems is a **greedy algorithm**: make the locally best choice at each step, in the hope that the steps will fit together into a solution that is not too far from the best solution. At each turn I will choose the step that minimizes the number of **excess letters** added to the path, and will never undo a step. Here is the exact definition of the metric we are trying to minimize:

- **Excess letters**: the number of letters that a step adds, relative to a baseline model in which all the words are concatenated with no repeated words and no overlap between them. (That's not a valid solution, but it is useful as a benchmark.) So if a step adds an unused word, and it woverlaps with the previous word by three letters, that is an excess of -3: I've saved three letters over just concatenating the unused word. (Note that a negative excess is a positive thing.) For a bridging word step, the excess is the number of letters that do not overlap either the previous word or the next word. So all unused word steps have negative excess and all bridging steps are non-negative. (Therefore if there is an unused word step, we don't need to consider a bridging step.)

**Examples:** In each row of the table below, `'ajar'` is the previous word,  but each row makes different assumptions about what unused words remain, and thus we get different choices for the step to take. The table shows the overlapping letters between the previous word and the step, and in the case of bridges, it shows one of the  possible unused words that the step is bridging to. The final column shows the excess letter score (and actual letters).

|Previous|Step(s)|Overlap|Bridge to|Type of Step(s)|Excess|
|--------|----|----|---|---|---|
| ajar|jarring|jar||*unused word* |-3| 
| ajar|arbitrary|ar||*unused word* |-2|
| ajar|rabbits|r||*unused word*|-1|
| ajar|argot|ar|goths|*one-step bridge* |0|
| ajar|arrow|ar|owlets| *one-step bridge*|1 (r)|
| ajar|rani, iraq|r|quizzed| *two-step bridge*|5 (anira) | 

Let's go over the examples:
- **jarring**: Here we assume `jarring` is an unused word. It overlaps with `ajar` by 3 letters, giving it an excess cost of -3.
- **arbitrary** and **rabbits**: unused word that overlap by fewer than 3 letters, so would only be chosen if there were no unused words with more overlap.
- **argot** and **arrow**: One-step bridges; a bridge with the least excess (non-overlapping letters) would be chosen.
- **rani, iraq**: a two-step bridge. Suppose `quizzed` is the only remaining unused word. There is no single word that bridges from any suffix of `ajar` to any prefix of `quizzed`. But `rani` can bridge from `'r'` to `'i'` and `iraq` can bridge from `'i'` to `'q'`. This two-word bridge has an excess score of 5 due to the letters `anira` not overlapping anything.

One optimization is to recognize that some words are **subwords** of other words. For example, `jar` is a subword of `ajar`. So anytime we place `ajar` into a step, we've also automatically placed `jar` there. We can save computation time by initializing the set of unused words to be the **nonsubwords** in *W*. A subword will never be added in an unused word step, but it may be used in a bridging word step. 

# Data Type Implementation

Here I describe how to implement the main data types in Python:

- **Word**: a Python `str` (as are subparts of words, like suffixes or individual letters).
- **Wordset**: a subclass of `set`, denoting a set of words, plus some cached attributes to be explained later.
- **Path**: a Python `list` of steps.
- **Step**: a named tuple of an overlap and a word. Following `ajar` with `jarring`  is `Step(3, 'jarring')`. 
- **Bridge**: a named tuple of an excess cost followed by a list of one or two steps, e.g. `Bridge(1, [Step(2, 'arrow')])`.
- **Bridges**: a table mapping a prefix and a suffix to a bridge. 

In [1]:
from collections import defaultdict, Counter, namedtuple

Word    = str
Step    = namedtuple('Step', 'overlap, word')
Bridge  = namedtuple('Bridge', 'excess, steps')
Path    = list[Step]
Bridges = dict[str, dict[str, Bridge]] # bridges[prefix][suffix] = Bridge(...)

class Wordset(set): 
    """A set of words, with slots to hold some cached information."""
    __slots__ = ('subwords', 'short_words', 'bridges', 'unused_words', 'unused_startswith')

# portman and is_portman

Here we define the functions `portman` and `is_portman`:

In [2]:
def portman(P: Path) -> Word:
    """Compute the portmantout string S from the path P."""
    # Concatenate the non-overlapping part of each step
    return ''.join(word[overlap:] for (overlap, word) in P)

def is_portman(P: Path, W: Wordset) -> bool:
    """Is the Path P a portmantout of the Wordset W?"""
    S = portman(P)
    return (all(word in S for word in W) and         # 1. Every word in W is a substring of S
            all(step.overlap > 0 for step in P[1:])) # 2. Every word (except first) overlaps

A tiny example wordset `W1`, path `P1`, and string `S1`:

In [3]:
W1 = Wordset({'anarchy', 'dashiki', 'grammarian', 'kimono', 'monogram',
              'a', 'am', 'an', 'arc', 'arch', 'aria', 'as', 'ash', 'dash', 'gram', 
              'grammar', 'i', 'mar', 'maria', 'mono', 'narc', 'no', 'on', 'ram'})

P1 = [Step(0, 'dashiki'),
      Step(2,      'kimono'),
      Step(4,        'monogram'),
      Step(4,            'grammarian'),
      Step(2,                    'anarchy')]

S1 = portman(P1)
S1

'dashikimonogrammarianarchy'

In [4]:
is_portman(P1, W1)

True

# natalie

The function `natalie` does a greedy search for a portmantout path. As stated above, the approach is to start with a path of one word (either given as an optional argument or chosen arbitrarily from the word set *W*), and then repeatedly adds steps, each step being either an `unused_word_step` or a sequence of one or two `bridging_steps`. The call `initialize(W)` does some computation on the word set `W` such as precomputing the set of possible bridges and setting `W.unused_words` to be the set of nonsubwords.

The function `unused_word_step` returns zero or one step, and `bridging_steps` returns one or two steps; I decided the simplest interface is to have them both return a list of steps.

In [5]:
def natalie(W: Wordset, start=None) -> Path:
    """Return a portmantout path containing all words in W. You can optionally give the start word."""
    initialize(W)
    first_word = start or first(W.unused_words)
    P = add_step([], W, Step(0, first_word))
    while W.unused_words:
        prev = P[-1].word
        steps = unused_word_step(W, prev) or bridging_steps(W, prev)
        for step in steps:
            P = add_step(P, W, step)
    return P

# Steps: unused_word_step and bridging_steps

`unused_word_step` considers every suffix of the previous word, where the function `suffixes` is guaranteed to order the longest suffixes first. If a suffix starts an unused words, we choose it. Since we're going longest-suffix first, no other word choice could do better on the excess letters metric.

In [6]:
def unused_word_step(W: Wordset, prev_word: Word) -> list[Step]:
    """Return Step(overlap, unused_word) or None."""
    for suffix in suffixes(prev_word):
        unused_word = first(W.unused_startswith.get(suffix, ()))
        if unused_word:
            return [Step(len(suffix), unused_word)]
    return []

(*Python trivia:* in `unused_word_step` I do `W.unused_startswith.get(suf, ())`, not `W.unused_startswith[suf]`  because the dict in question is a `defaultdict`, and if there is no entry there, I don't want to insert a default entry.)

`bridging_steps` also considers every suffix of the previous word, and for each one it looks in the `W.bridges[suf]` table (see below) to see what prefixes (of unused words) we can bridge to from this suffix. Out of all the bridges in `W.bridges[suf][pre]` entries, take one with the minimal excess cost, and return the steps that make up that bridge.

In [7]:
def bridging_steps(W: Wordset, prev_word: Word) -> list[Step]:
    """The steps from the minimal-excess bridge that bridges 
    from a suffix of prev_word to a prefix of any unused word."""
    bridges = [W.bridges[suf][pre] 
               for suf in suffixes(prev_word) if suf in W.bridges
               for pre in W.bridges[suf] if W.unused_startswith[pre]]
    return min(bridges).steps # Choose the bridge with minimal excess

# initialize and add_step

To make it efficient to find steps, the function  `initialize(W)` caches the following information on `W`:
  - `W.unused_words`: initially the set of nonsubwords in `W`; when a word is used it is removed from the set.
  - `W.unused_startswith`: a dict that maps from a prefix to all the unused words that start with the prefix. Updated when a word is used.
      - Example: `W.unused_startswith['somet'] == {'something', 'sometimes'}`.
  - `W.subwords`: a set of all the words that are contained within another word in `W`.
  - `W.bridges`: a dict where `W.bridges[suf][pre]` gives the best bridge between the affixes.
  - `W.short_words`: a set of short words used to build bridges. (See `build_bridges`.)

  
These structures are complicated, so don't be discouraged if you have to go over the code several times. 

In [8]:
def initialize(W: Wordset) -> Wordset:
    """Precompute and cache data structures on attributes of W."""
    # Initialize .bridges and .subwords only once; they're unchanging
    if not hasattr(W, 'bridges'): 
        W.bridges       = build_bridges(W)
        W.subwords      = {subword for w in W for subword in subparts(w) & W}     
    # Re-initialize .unused_words and .unused_startswith every time we want to generate a portmantout
    # They are updated every time we add a step to a path
    W.unused_words      = W - W.subwords   
    W.unused_startswith = startswith_table(W.unused_words)

    return W

Now we can add a step: append the step to the path we are building up, remove the word from the unused words, and remove the word from all the places where it is stored as an unused word with a given prefix.

In [9]:
def add_step(P: Path, W: Wordset, step: Step) -> Path:
    """Add step to P; remove step's word from `W.unused_words` and `W.unused_startswith[pre] for each pre`."""
    P.append(step)
    word = step.word
    if word in W.unused_words: # Maintain W.unused_words and W.unused_startswith
        W.unused_words.remove(word)
        for pre in prefixes(word):
            W.unused_startswith[pre].remove(word)
            if not W.unused_startswith[pre]: # clean up
                del W.unused_startswith[pre]
    return P

# Building Bridges

The last major piece of the program is the construction of the `W.bridges` table. Recall that we want `W.bridges[suf][pre]` to be a bridge between a suffix of the previous word and a prefix of a nonsubword, as in the examples:


      W.bridges['ar']['ow'] == Bridge(1, [Step(2, 'arrow')])
      W.bridges['ar']['c']  == Bridge(0, [Step(2, 'arc')])
      W.bridges['r']['q']   == Bridge(5, [Step(1, 'rani'), Step(1, 'iraq')])
      
We build all the bridges in `initialize`, and don't update them. Thus, `W.bridges['r']['q']` says "if the previous word ends in `'r'` and there is an unused words starting with `'q'`, you can use this bridge." The caller is responsible for checking that `W.unused_startswith['q']` contains unused word(s).
      
Bridges should be short (they should have a small excess letter count). We don't need to consider `antidisestablishmentarianism` as a possible bridge word. Instead, from our 108,709  word set *W*, we'll define `W.short_words` to be a set of 11,330 words that are no more than a limit of 5 letters long, except that we add one letter to the limit if the first letter is a rare one, and add one more if the last letter is a rare one.  I also compute a `short_startswith` table for the `short_words`, where, for example,

     short_startswith['som'] == {'soma', 'somas', 'some'} # but not 'somebodies', 'something', ...

To build one-word bridges, consider every short word, and split it up in all possible ways into a prefix that will overlap the previous word, a suffix that will overlap the next word, and a count of zero or more excess letters in the middle that don't overlap anything. The function `splits` helps:

In [10]:
def splits(word: Word) -> list[tuple[str, int, str]]: 
    """Ways to split up word, as a list of (prefix, excess, suffix) tuples."""
    return [(word[:i], excess, word[i+excess:])
            for excess in range(len(word) - 1)
            for i in range(1, len(word) - excess)]

def is_short_word(word: Word, maxlen=5, rare_start='jkqxyz', rare_end='bfikopquvwxz') -> bool: 
    """Is this a short word, suitable for use in bridges?"""
    return len(word) <= maxlen + int(word[0] in rare_start) + int(word[-1] in rare_end)

For example:

In [11]:
splits('arrow')

[('a', 0, 'rrow'),
 ('ar', 0, 'row'),
 ('arr', 0, 'ow'),
 ('arro', 0, 'w'),
 ('a', 1, 'row'),
 ('ar', 1, 'ow'),
 ('arr', 1, 'w'),
 ('a', 2, 'ow'),
 ('ar', 2, 'w'),
 ('a', 3, 'w')]

The first element of the list says that `'arrow'` can bridge from `'a'` to `'rrow'` with 0 excess letters; the last says it can bridge from `'a'` to `'w'` with 3 excess letters (which happen to be `'rro'`). 

Each possible split is passed on to `build_bridge`, which records a bridge in the table under `bridges[pre][suf]` unless there is already ridge stored there that has a smaller ecess letter count.

In [12]:
def build_bridge(bridges: Bridges, word: Word, pre: str, excess: int, suf: str, step2=None):
    """Store a new bridge (of 1 or 2 steps) if it has less excess than the previous bridges[pre][suf]."""
    if suf not in bridges[pre] or excess < bridges[pre][suf].excess:
        steps = [Step(len(pre), word)]
        if step2: steps.append(step2)
        bridges[pre][suf] = Bridge(excess, steps)

It is an unfortunate fact that if we only allow one-word bridges, the algorithm can get stuck in a dead end. But if we allow all possible two-word bridges the algorithm would be slow, bogged down with all those bridges to check. Thus, I decided to only use two-word bridges that bridge from a single last letter in the previous word to a single first letter in a following word. If we can construct a bridge from every single letter to every other single letter, then we know the algorithm can never get stuck.

Here's the complete `build_bridges` function:

In [13]:
def build_bridges(W: Wordset):
    """A table of bridges[pre][suf] == Bridge(excess, [Step(overlap, word)]), e.g.
    bridges['ar']['c'] == Bridge(0, [Step(2, 'arc')]).
    bridges['d']['z']  == Bridge(3, [Step(1, 'do'), Step(1, 'oyez')])"""
    W.short_words    = set(filter(is_short_word, W))
    short_startswith = startswith_table(W.short_words)
    bridges          = defaultdict(dict)
    # One-word bridges: every way to split up every short word into suffix/excess/prefix
    for word in W.short_words: 
        for split in splits(word):
            build_bridge(bridges, word, *split)
    # Two-word bridges: only bridge from a single letter to a single letter
    for word1 in W.short_words:
        for suf in suffixes(word1): 
            for word2 in short_startswith[suf]: 
                excess = len(word1) + len(word2) - len(suf) - 2
                A, B = word1[0], word2[-1] # First and last letters
                if A != B: # No sense bridging from A to A
                    step2 = Step(len(suf), word2)
                    build_bridge(bridges, word1, A, excess, B, step2)
    return bridges

# Utility Functions

In [14]:
def multimap(pairs) -> dict[object, set]:
    """Given (key, val) pairs, make a dict of {key: {val,...}}."""
    result = defaultdict(set)
    for key, val in pairs:
        result[key].add(val)
    return result

def startswith_table(words) -> dict[str, set[Word]]: 
    """A dict mapping a prefix to all the words it starts:
    {'somet': {'something', 'sometimes'},...}."""
    return multimap((pre, w) for w in words for pre in prefixes(w))   
    
def suffixes(word: Word) -> list[str]:
    """All non-empty proper suffixes of word, longest first."""
    return [word[i:] for i in range(1, len(word))]

def prefixes(word: Word) -> list[str]:
    """All non-empty proper prefixes of word."""
    return [word[:i] for i in range(1, len(word))]

def subparts(word: Word) -> set[str]:
    """All non-empty proper substrings of word"""
    return {word[i:j] 
            for i in range(len(word)) 
            for j in range(i + 1, len(word) + 1)} - {word}

def first(iterable) -> object | None:
    """The first element in an iterable, or None"""
    return next(iter(iterable), None)

(*Math trivia:* In this context, "proper" means "not whole". A proper subset is a subset that is not the whole set itself; a proper substring of a word is a substring that is not the whole word itself.)

# W: Tom Murphy's Wordset 

We can make Tom Murphy's 108,709 word file `"wordlist.asc"` into a `Wordset` called `W`:

In [15]:
! [ -e wordlist.asc ] || curl -O https://norvig.com/ngrams/wordlist.asc

W = Wordset(open('wordlist.asc').read().split()) 
assert len(W) == 108709

In [16]:
!head wordlist.asc

a
aahed
aahing
aardvark
aardvarks
aardwolf
abaci
aback
abacus
abacuses


# Portmantout Solutions

**Finally!** We're ready to make portmantouts. First for the tiny word set `W1`, for which we must carefully choose the starting word:

In [17]:
P1 = natalie(W1, start='dashiki')
P1

[Step(overlap=0, word='dashiki'),
 Step(overlap=2, word='kimono'),
 Step(overlap=4, word='monogram'),
 Step(overlap=4, word='grammarian'),
 Step(overlap=2, word='anarchy')]

In [18]:
portman(P1)

'dashikimonogrammarianarchy'

Now for the big word set `W`:

In [19]:
%time P = natalie(W)
len(P)

CPU times: user 5.45 s, sys: 34 ms, total: 5.49 s
Wall time: 5.5 s


103069

I thought it might take several minutes, so 5 seconds to find a 100,000+ step path is great! Now to generate (but not print) the portmantout string:

In [20]:
S = portman(P)
len(S)

553434

 The string is about a half-million letters long.

# Failure is Not an Option

Is `natalie` guaranteed to terminate with a solution in finite time? Every iteration either uses up an unused word, or builds a bridge to an unused word that will be used on the next iteration. So, eventually all the unused words will be used and `natalie` will return a solution. The only way this can fail is if we get stuck in a situation where there is no bridge to an unused word. I can prove that this can't happen if I can verify that there is a bridge from every one-letter suffix to every one-letter prefix. The function `missing_bridges` checks for this.

In [21]:
alphabet     = 'abcdefghijklmnopqrstuvwxyz'
letter_pairs = [A + B for A in alphabet for B in alphabet if A != B]

def missing_bridges(W: Wordset) -> list[str]:
    """What 1-letter-suffix to 1-letter-prefix bridges are missing from W.bridges?"""
    return [A + B for (A, B) in letter_pairs if B not in W.bridges[A]]

Great! *W* has no missing bridges. But the tiny *W1* is missing 623 out of 26 × 25 = 650 1-to-1-letter bridges:

In [22]:
len(missing_bridges(initialize(W1)))

621

# Pretty Output

Notice I haven't actually *looked* at the portmantout yet. I didn't want to dump half a million letters into an output cell. Instead, I'll define `report` to print various statistics, summarize the begin and end of the portmantout, and save the full string *S* into a file. 

In [23]:
def report(W: Wordset, P: Path, steps=100, letters=1000, save='natalie.txt'):
    S       = portman(P)
    sub     = W.subwords 
    nonsub  = W - sub
    uniq    = {step.word for step in P} # unique step words in P
    bridge  = len(P) - len(nonsub) # number of bridge steps in P
    bridges = sum(len(W.bridges[pre]) for pre in W.bridges) # number of bridges in W
    def L(words) -> int: return sum(map(len, words)) # Number of letters
    print(f'W has {len(W):,d} words ({len(nonsub):,d} nonsubwords; {len(sub):,d} subwords).')
    print(f'W has {bridges:,d} bridges from {len(W.short_words):,d} short_words, '
          f'and {len(missing_bridges(W))} missing 1-to-1-letter bridges.')
    print(f'P has {len(P):,d} steps ({len(uniq):,d} unique words; {bridge:,d} bridge words).')
    print(f'P has an average overlap of {(L(s.word for s in P)-len(S))/(len(P)-1):.2f} letters.')
    print(f'P (and thus S) is {"" if is_portman(P, W) else "NOT "}a valid portmantout of W.')
    print(f'S has a compression ratio (letters(W)/letters(S)) of {L(W)/len(S):.2f}.')
    print(f'S has {len(S):,d} letters; W has {L(W):,d}; nonsubs have {L(nonsub):,d}.')
    if save: open(save, "w").write(S)
    print(f'S saved as the file "{save}".')
    print(f'\nThe first and last {letters} letters:\n\n{S[:letters]}\n...\n{S[-letters:]}')
    steps1 = ', '.join(w[:i] + '⋅' + w[i:] for i, w in P[:steps])
    steps2 = ', '.join(w[:i] + '⋅' + w[i:] for i, w in P[-steps:])
    print(f'\nThe first and last {steps} steps:\n\n{steps1}\n...\n{steps2}')

A step such as `Step(1, 'sir')` is printed as `s⋅ir` to indicate that `s` is the 1-letter overlap with the previous word.

I will redefine `is_portman` to be faster. *Python trivia:* if `X, Y` and `Z` are sets, `X <= Y <= Z` means "is `X` a subset of `Y` and `Y` a subset of `Z`?" We use the notation here to say that the set of words in *P* must contain all the nonsubwords and can only contain words from *W*.

In [24]:
def is_portman(P: Path, W: Wordset) -> str:
    """Verify that P forms a valid portmantout string for W."""
    uses_right_words = (W - W.subwords) <= set(step.word for step in P) <= W
    overlaps_match   = all((overlap > 0 and P[i - 1].word[-overlap:] == word[:overlap])
                           for i, (overlap, word) in enumerate(P[1:], 1))
    return uses_right_words and overlaps_match and P[0].overlap == 0

In [25]:
report(W, P)

W has 108,709 words (64,389 nonsubwords; 44,320 subwords).
W has 68,178 bridges from 11,330 short_words, and 0 missing 1-to-1-letter bridges.
P has 103,069 steps (65,059 unique words; 38,680 bridge words).
P has an average overlap of 1.65 letters.
P (and thus S) is a valid portmantout of W.
S has a compression ratio (letters(W)/letters(S)) of 1.68.
S has 553,434 letters; W has 931,823; nonsubs have 595,805.
S saved as the file "natalie.txt".

The first and last 1000 letters:

demultiplexeskimosquitoeshoestringsidespinieruptionshorelessenedematadorsalsifyarestyledwardshipsidestrokestrelsewherefromagesturersatzestfulnessesamescalismithyselfhoodsociologicallylsympathizingynecologiestonianswerabilityphusesquicentenniallymphocyteslasherselfingerboardspeediestockshrivelledentatestimoniescalopediatricianswererstwhileditoriallynchingsubscriptediousnessayersagiestoppingstomachicallymphaticallymphoidiumlauteddiesestinessentiallyricallyratediumsupplementarythmiasmatavistsarismsalutarythmicroprogr

# Questions

The program is complete, but there are still many interesting things to explore, and questions to answer.

**Question: is there an imbalance in starting and ending letters of words?** That could lead to a need for many bridges. We saw in the last 100 steps of *P* multiple repetitions of the two-word bridge "s⋅ir, ir⋅aq". That suggests there are too many words that end in "s" and too many that start with "q". Let's investigate:

In [26]:
initialize(W)

starts = Counter(w[0]  for w in W.unused_words)
ends   = Counter(w[-1] for w in W.unused_words)

def ratio(L: str) -> str:
    """Approximate ratio of words that start with L to words that end with L."""
    s, e = starts[L], ends[L]
    return f'{round(s/e)}:1' if (s > e and e != 0) else f'1:{round(e/s)}'

print('Letter Starts   Ends Ratio')
print('------ ------ ------ -----')
for L in sorted(starts):
    print(f'{L:>5}  {starts[L]:6,d} {ends[L]:6,d} {ratio(L):>5}')

Letter Starts   Ends Ratio
------ ------ ------ -----
    a   3,528    384   9:1
    b   3,776      6 629:1
    c   5,849    908   6:1
    d   4,093  7,520   1:2
    e   2,470  3,215   1:1
    f   2,794     51  55:1
    g   2,177  6,343   1:3
    h   2,169    351   6:1
    i   2,771    128  22:1
    j     638      0   1:0
    k     566    157   4:1
    l   1,634  1,182   1:1
    m   3,405    657   5:1
    n   1,542  1,860   1:1
    o   1,797    113  16:1
    p   4,977    123  40:1
    q     330      0   1:0
    r   3,811  1,994   2:1
    s   7,388 29,056   1:4
    t   3,097  2,107   1:1
    u   2,557     11 232:1
    v   1,032      6 172:1
    w   1,561     42  37:1
    x      51     68   1:1
    y     207  8,086  1:39
    z     169     21   8:1


Yes, there is a problem: there are many more words that start with `b`, `f`, `p`, `u`, `u` and `v` than that end with those letters. In the other direction 45% of all words end in `s`, but only a quarter of that number start with `s`. The start:end ratio for `y` is 1:39.

In [27]:
ends['s'] / len(W.unused_words)

0.451257202317166

**Question: what are the most common words in path *P*?** 

These will be bridge words. What do they have in common?

In [28]:
Counter(step.word for step in P).most_common(12)

[('sap', 2545),
 ('so', 2319),
 ('sic', 1793),
 ('of', 1792),
 ('lyre', 1685),
 ('dab', 1596),
 ('gab', 1515),
 ('sun', 1495),
 ('sin', 1429),
 ('yam', 1294),
 ('saw', 1000),
 ('lye', 713)]

Indeed,  bridging away from `s` is a big concern (half of the top dozen bridges). Also, `lyre` and `lye` bridge away from an adverb ending, `ly` (as can `yep`).

I'm surprised that `of` shows up so frequently. Let's see what it is bridging from:

In [29]:
Counter(P[i-1].word for i, step in enumerate(P) if step.word == 'of').most_common()

[('so', 1327),
 ('go', 270),
 ('do', 180),
 ('to', 8),
 ('cairo', 1),
 ('furioso', 1),
 ('francisco', 1),
 ('franco', 1),
 ('fortissimo', 1),
 ('vulgo', 1),
 ('fresno', 1)]

We see that `of` is used in two-word bridges to go from `s`, `g`, and  `d` to `f`. 

**Question: What is the distribution of word lengths?** 

In [30]:
Counter(map(len, W.unused_words)) # Counter of word lengths

Counter({8: 11964,
         9: 11950,
         7: 8672,
         10: 8443,
         11: 6093,
         12: 4423,
         6: 4364,
         13: 2885,
         5: 1796,
         14: 1765,
         15: 1017,
         16: 469,
         17: 198,
         4: 186,
         18: 91,
         19: 33,
         20: 22,
         21: 9,
         22: 4,
         23: 2,
         3: 2,
         28: 1})

**Question: What is the longest word?** 

In [31]:
max(W, key=len)

'antidisestablishmentarianism'

**Question: What is the distribution of letters in the Wordset?**

In [32]:
Counter(L for w in W.unused_words for L in w).most_common() # Counter of letters

[('e', 68038),
 ('s', 60080),
 ('i', 53340),
 ('a', 43177),
 ('n', 42145),
 ('r', 41794),
 ('t', 38093),
 ('o', 35027),
 ('l', 32356),
 ('c', 23100),
 ('d', 22448),
 ('u', 19898),
 ('g', 17815),
 ('p', 16128),
 ('m', 16062),
 ('h', 12673),
 ('y', 11889),
 ('b', 11581),
 ('f', 7885),
 ('v', 5982),
 ('k', 4892),
 ('w', 4880),
 ('z', 2703),
 ('x', 1677),
 ('j', 1076),
 ('q', 1066)]

**Question: How many bridges are there?** 

In [33]:
# Make a list of all bridges, B
B = [W.bridges[suf][pre] for suf in W.bridges for pre in W.bridges[suf]]
len(B)

68178

In [34]:
B[::2000] # Sample every 2000th bridge

[Bridge(excess=0, steps=[Step(overlap=1, word='tauts')]),
 Bridge(excess=1, steps=[Step(overlap=1, word='seers')]),
 Bridge(excess=1, steps=[Step(overlap=1, word='wasps')]),
 Bridge(excess=2, steps=[Step(overlap=1, word='hiccup')]),
 Bridge(excess=1, steps=[Step(overlap=1, word='jell')]),
 Bridge(excess=0, steps=[Step(overlap=1, word='mopy')]),
 Bridge(excess=0, steps=[Step(overlap=1, word='buxom')]),
 Bridge(excess=1, steps=[Step(overlap=2, word='doffs')]),
 Bridge(excess=0, steps=[Step(overlap=1, word='cumin')]),
 Bridge(excess=1, steps=[Step(overlap=1, word='fade')]),
 Bridge(excess=0, steps=[Step(overlap=1, word='gazebo')]),
 Bridge(excess=0, steps=[Step(overlap=1, word='nebs')]),
 Bridge(excess=0, steps=[Step(overlap=2, word='tees')]),
 Bridge(excess=0, steps=[Step(overlap=1, word='imams')]),
 Bridge(excess=0, steps=[Step(overlap=3, word='gene')]),
 Bridge(excess=0, steps=[Step(overlap=3, word='workup')]),
 Bridge(excess=0, steps=[Step(overlap=1, word='view')]),
 Bridge(excess=0, 

**Question: How many excess letters do the bridges have?** 

In [35]:
# Counter of bridge excess letters
BC = Counter(b.excess for b in B)
BC

Counter({0: 42395, 1: 20588, 2: 4539, 3: 578, 4: 50, 5: 21, 6: 6, 8: 1})

Most of the bridges have 0 or 1 excess letter, so we're doing pretty well.

In [36]:
from statistics import mean

mean(BC.elements())

0.46567807797236643

**Question: How many 1-step and 2-step bridges are there?**

In [37]:
Counter(len(b.steps) for b in B)

Counter({1: 68033, 2: 145})

There are only 148 2-step bridges; we might as well see them all:

In [38]:
{A + B: W.bridges[A][B] for A, B in letter_pairs
 if len(W.bridges[A][B].steps) == 2}

{'af': Bridge(excess=2, steps=[Step(overlap=1, word='ago'), Step(overlap=1, word='of')]),
 'ag': Bridge(excess=2, steps=[Step(overlap=1, word='an'), Step(overlap=1, word='nag')]),
 'aj': Bridge(excess=4, steps=[Step(overlap=1, word='ash'), Step(overlap=1, word='hadj')]),
 'aq': Bridge(excess=3, steps=[Step(overlap=1, word='air'), Step(overlap=2, word='iraq')]),
 'av': Bridge(excess=1, steps=[Step(overlap=1, word='at'), Step(overlap=1, word='tv')]),
 'bc': Bridge(excess=2, steps=[Step(overlap=1, word='bar'), Step(overlap=2, word='arc')]),
 'bj': Bridge(excess=4, steps=[Step(overlap=1, word='bacon'), Step(overlap=3, word='conj')]),
 'bq': Bridge(excess=6, steps=[Step(overlap=1, word='belli'), Step(overlap=1, word='iraq')]),
 'bv': Bridge(excess=2, steps=[Step(overlap=1, word='but'), Step(overlap=1, word='tv')]),
 'cv': Bridge(excess=2, steps=[Step(overlap=1, word='cot'), Step(overlap=1, word='tv')]),
 'df': Bridge(excess=1, steps=[Step(overlap=1, word='do'), Step(overlap=1, word='of')]),

**Question: What strange letter combinations are there?** Let's look at two-letter suffixes or prefixes that only appear in one or two nonsubwords. 

In [39]:
{pre: W.unused_startswith[pre] # Rare two-letter prefixes
 for pre in letter_pairs if len(W.unused_startswith[pre]) in (1, 2)}

{'aj': {'ajar'},
 'ay': {'ayahs', 'ayatollahs'},
 'bw': {'bwanas'},
 'ct': {'ctrl'},
 'dn': {'dnieper'},
 'dv': {'dvorak'},
 'ek': {'ekistics'},
 'ez': {'ezekiel'},
 'fb': {'fbi'},
 'fj': {'fjords'},
 'gj': {'gjetosts'},
 'gw': {'gweducks', 'gweducs'},
 'hd': {'hdqrs'},
 'ie': {'ieee'},
 'if': {'iffiness'},
 'ik': {'ikebanas', 'ikons'},
 'ip': {'ipecacs'},
 'iv': {'ivories', 'ivory'},
 'jn': {'jnanas'},
 'kw': {'kwachas', 'kwashiorkor'},
 'mc': {'mcdonald'},
 'oj': {'ojibwas'},
 'pf': {'pfennigs'},
 'qa': {'qaids', 'qatar'},
 'qo': {'qophs'},
 'sf': {'sforzatos'},
 'tc': {'tchaikovsky'},
 'uf': {'ufos'},
 'wu': {'wurzel'},
 'xi': {'xiphoids', 'xiphosuran'},
 'xm': {'xmases'},
 'yc': {'ycleped', 'yclept'},
 'ym': {'ymca'},
 'zl': {'zlotys'},
 'zw': {'zwiebacks'}}

In [40]:
endswith = multimap((w[-2:], w) for w in W.unused_words)

{suf: endswith[suf] # Rare two-letter suffixes
 for suf in endswith if len(endswith[suf]) <= 2}

{'mb': {'clomb', 'whitecomb'},
 'dn': {'haydn'},
 'rf': {'waldorf', 'windsurf'},
 'ao': {'chiao', 'ciao'},
 'gn': {'champaign'},
 'zm': {'transcendentalizm'},
 'gm': {'apophthegm'},
 'sr': {'ussr'},
 'nu': {'vishnu'},
 'ku': {'haiku'},
 'ec': {'filespec', 'quebec'},
 'lu': {'honolulu'},
 'we': {'zimbabwe'},
 'ua': {'joshua'},
 'nx': {'bronx', 'meninx'},
 'tl': {'peyotl', 'shtetl'},
 'mt': {'daydreamt', 'undreamt'},
 'cd': {'recd'},
 'hr': {'kieselguhr'},
 'ui': {'maqui', 'prosequi'},
 'zo': {'diazo', 'palazzo'},
 'bm': {'ibm', 'icbm'},
 'su': {'shiatsu'},
 'ud': {'aloud', 'overproud'},
 'aa': {'markkaa'},
 'ji': {'fiji'},
 'nc': {'dezinc', 'quidnunc'},
 'mp': {'prestamp'},
 'wa': {'kiowa', 'okinawa'},
 'ou': {'thankyou'},
 'xe': {'deluxe', 'maxixe'},
 'oz': {'kolkhoz'},
 'ko': {'gingko', 'stinko'},
 'xo': {'convexo'},
 'xs': {'duplexs'},
 'ob': {'blowjob'},
 'za': {'organza'},
 'pa': {'tampa'},
 'ho': {'groucho'},
 'nz': {'franz'},
 'sz': {'grosz'},
 'td': {'retd'},
 'ab': {'skylab'},


The two-letter prefixes definitely include some strange words.

The list of two-letter suffixes is mostly picking out proper names and pointing out flaws in the word list. For example, lots of words end in `ab`: blab, cab, crab, dab, gab, jab, lab, etc. But must of them are subwords of plural forms; only `skylab` made it into the word list in singular form but not plural.

# Comparison to Tom Murphy's Program

To compare my [program](portman.py) to [Tom Murphy's](https://sourceforge.net/p/tom7misc/svn/HEAD/tree/trunk/portmantout/): 
- My string over *W* is about 554,000 letters; Murphy's is 611,000.
- I used a greedy approach that builds up a single long portmanteau, one step at a time. 
- Murphy first built a pool of smaller portmanteaux, then greedily joined them all together.
- I used Python (about 125 lines for the program without the exploratory questions and the pretty output).
- Murphy used C++ (1867 lines), with a lot of extra functionality I didn't do: generating diagrams and animations, and running multiple threads in parallel.

I'm reminded of the [Traveling Salesperson Problem](TSP.ipynb) where one algorithm is to form a single path, always progressing to the nearest neighbor, and another algorithm is to maintain a pool of shorter segments and repeatedly join together the two closest segments. The two approaches are different, but they are both suboptimal greedy methods, andit is not clear whether one is better than the other. You could try it!

(*English trivia:*  my program builds a single path of words, and when the path gets stuck and I need something to allow me to continue, it makes sense to call that thing a **bridge**.  Murphy's program starts by building a large pool of small portmanteaux that he calls **particles**, and when he can build no more particles, his next step is to put two particles together, so he calls it a **join**. The different metaphors for what our programs are doing lead to different terminology for the same idea.)


 

It appears Murphy  perhaps didn't quite have the complete concept of **subwords**. He did mention that when he adds `'bulleting'`, he crosses `'bullet'` and `'bulletin'` off the list, but somehow  [his string](http://tom7.org/portmantout/murphy2015portmantout.pdf) contains both `'spectacular'` and `'spectaculars'`. My guess is that if he adds `'spectaculars'` first he crosses off `'spectacular'`, but if he happens to add `'spectacular'` first, he will later add `'spectaculars'`. Support for this view is that his output in `bench.txt` says "I skipped 24319 words that were already substrs." but I computed that there are 44,320 such subwords; he found about half of them. I think those missing 20,001 words are the main reason why my strings are shorter.

Also, Murphy's joins are always between one-letter prefixes and suffixes. I allow prefixes and suffixes of any length up to a total of 6 for `len(pre) + len(suf)`, for one-word bridges. I can get away with this because I limited my candidate pool to the 10,000 `W.short_words`. It would have been time-consuming to build all bridges for all 100,000 words, and probably would not have helped shorten *S* appreciably.

I should say that I stole one important trick from Murphy. After I finished the first version of my program, I looked at his highly-entertaining [video](https://www.youtube.com/watch?time_continue=1&v=QVn2PZGZxaI) and [paper](http://tom7.org/portmantout/murphy2015portmantout.pdf) and I noticed that I had a problem in my use of bridges. My `natalie` function originally contained something like this: 

    unused_word_step(...) or one_word_bridge(...) or two_word_bridge(...)
    
That is, I only considered two-word bridges when there was no one-word bridge, on the assumption that one word is shorter than two. But Murphy showed that my assumption was wrong: for `bridges['w']['c']` I had `'workaholic'` as the best one-word bridge, but he had the two-word bridge `'war' + 'arc' = 'warc'`, which saves six excess letters over my single word. After seeing that, I shamelessly copied his approach, and now I too get a two-letter excess for `bridges['w']['c']`. (Sometimes  `'war' + 'arc'` and sometimes `'wet' + 'etc'` or `'we' + 'etc'`, depending on the seed for the hash function that hashes strings into a word set.)

In [41]:
W.bridges['w']['c']

Bridge(excess=2, steps=[Step(overlap=1, word='war'), Step(overlap=2, word='arc')])

# Conclusion

I'll stop here, but you should feel free to do more experimentation of your own. 

Here are some things you could do to make the portmantouts more interesting:

- Use linguistic resources (such as [pretrained word embeddings](https://nlp.stanford.edu/projects/glove/)) to teach your program what words are related to each other. Encourage the program to place  related words next to each other. Maybe even make grammatical sentences.
- Use linguistic resources (such as [NLTK](https://github.com/nltk/)) to teach your program where syllable breaks are in words, and what each syllable sounds like. Encourage the program to make overlaps match syllables. (That's why "preferendumdums" sounds better than "fortyphonshore".)

Here are some things you could do to make *S* shorter:

- **Lookahead**: Unused words are chosen based on the degree of overlap, but nothing else. It might help to prefer unused words which have a suffix that matches the prefix of another unused word. A single-word lookahead or a beam search could be used.
- **Reserving words**: It seems like `haydn` and `dnieper` are made to go together in that order; they're the only two words with `dn` as an affix. Similarly, `womenfolk` should be followed by `menfolks`. But if we happened to place `dnieper` or `menfolks` first, we would loose the chance of these nice overlaps.  Maybe there could be a system that assures the proper ordering, or a preprocessing step that joins together words that go together uniquely well. 
- **Word choice ordering**: Perhaps `startswith_table` could sort the words in each key's bucket so that the "difficult" words (say, the ones that end in unusual letters) are encountered earlier in the program's execution, when there are more available words for them to connect to.
- **Learning**: The greedy approach minimizes the number of excess letters for each step. But some words are harder to place than others. Instead of just minimizing the excess, consider also the *expected* excess of each word, which could be learned. 
  
Here are some things you could do to make the program more robust:

- Write and run unit tests.
- Find other word lists, perhaps in other languages, and try the program on them.
- Try word lists such as a list of names, or cities or countries, augmented with common short words.
- Consider what to do for a wordset that has missing bridges. You could try three-word bridges, you could allow the program to back up and remove a previously-placed word; you could allow the addition of words to the start as well as the end of `P`.